# Music Genre CNN: Explained from First Principles

This notebook explains the main parts of a Convolutional Neural Network (CNN) using the local GTZAN spectrogram images. The target is ten music genres. The FMA section near the end shows how its audio and metadata fit the same workflow: audio is converted into a mel spectrogram before entering the CNN.

The notebook uses PyTorch and torchvision. Accuracy is measured on a held-out test set; it is not assumed in advance.

In [ ]:
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "raw").exists() and (PROJECT_ROOT.parent / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
IMAGE_ROOT = PROJECT_ROOT / "raw" / "gtzan" / "Data" / "images_original"
GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]

## 1. Import Libraries and Configure Runtime

PyTorch supplies tensors, automatic differentiation, and neural-network layers. `torchvision` supplies a folder-based image dataset and transforms. A fixed seed makes the split and initialization repeatable. The device selects a CUDA GPU when one is available.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((128, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
dataset = datasets.ImageFolder(IMAGE_ROOT, transform=transform)
print("Classes:", dataset.classes)
print("Images:", len(dataset))
print("Class counts:", {name: sum(label == index for _, label in dataset.samples) for index, name in enumerate(dataset.classes)})

figure, axes = plt.subplots(2, 5, figsize=(15, 5))
for axis, index in zip(axes.flat, np.linspace(0, len(dataset) - 1, 10, dtype=int)):
    image, label = dataset[index]
    axis.imshow(image.permute(1, 2, 0).numpy() * 0.5 + 0.5)
    axis.set_title(dataset.classes[label])
    axis.axis("off")
plt.tight_layout()

## 2. Load and Inspect the Image Dataset

Each GTZAN image is a visual representation of an audio clip. The folder name is the label, so `ImageFolder` maps each genre to an integer. Resize gives every example the same height and width; normalization keeps values in a stable range for optimization.

In [ ]:
train_size = int(0.70 * len(dataset))
validation_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - validation_size
train_set, validation_set, test_set = random_split(
    dataset, [train_size, validation_size, test_size],
    generator=torch.Generator().manual_seed(SEED),
)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=0)
validation_loader = DataLoader(validation_set, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=0)
images, labels = next(iter(train_loader))
print("Batch images:", images.shape)
print("Batch labels:", labels.shape)

## 3. Preprocess Data and Create Batches

A batch is a small group of examples processed together. Shuffling the training set reduces ordering effects. Validation and test sets stay ordered and are used only for measurement.

In [ ]:
sample = torch.randn(4, 3, 128, 256)
for kernel_size, stride, padding in [(3, 1, 0), (3, 2, 1), (5, 1, 2)]:
    layer = nn.Conv2d(3, 8, kernel_size=kernel_size, stride=stride, padding=padding)
    print(f"kernel={kernel_size}, stride={stride}, padding={padding} -> {layer(sample).shape}")

## 4. Convolution Layer: Kernels, Stride, and Padding

A convolution kernel is a small learnable filter that slides over height and width. More output channels mean more learned feature detectors. Stride controls the step size, while padding adds a border so spatial dimensions can be preserved.

In [ ]:
conv_output = nn.Conv2d(3, 8, kernel_size=3, padding=1)(sample)
relu_output = nn.ReLU()(conv_output)
print("Negative values before ReLU:", (conv_output < 0).sum().item())
print("Negative values after ReLU:", (relu_output < 0).sum().item())
plt.hist(conv_output.detach().flatten().numpy(), bins=40, alpha=0.6, label="before ReLU")
plt.hist(relu_output.detach().flatten().numpy(), bins=40, alpha=0.6, label="after ReLU")
plt.legend(); plt.title("ReLU clips negative activations"); plt.show()

## 5. Activation Functions: ReLU

ReLU keeps positive values and replaces negative values with zero. This adds non-linearity, allowing stacked convolution layers to learn more than a single linear transformation.

In [ ]:
feature_map = torch.arange(64.0).reshape(1, 1, 8, 8)
max_pool = nn.MaxPool2d(2)(feature_map)
avg_pool = nn.AvgPool2d(2)(feature_map)
print("Input:", feature_map.shape)
print("MaxPool:", max_pool.shape)
print("AvgPool:", avg_pool.shape)
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for axis, value, title in zip(axes, [feature_map, max_pool, avg_pool], ["Input", "MaxPool", "AvgPool"]):
    axis.imshow(value[0, 0].numpy(), cmap="magma"); axis.set_title(title); axis.axis("off")
plt.show()

## 6. Pooling Layers: MaxPool and AvgPool

Pooling downsamples feature maps. Max pooling keeps the strongest response in each window; average pooling keeps the mean response. Both reduce computation and make features less sensitive to small shifts.

In [ ]:
pooled = nn.MaxPool2d(2)(conv_output)
flattened = torch.flatten(pooled, start_dim=1)
linear = nn.Linear(flattened.shape[1], len(GENRES))
logits = linear(flattened)
print("Feature map:", conv_output.shape)
print("Pooled feature map:", pooled.shape)
print("Flattened:", flattened.shape)
print("Class logits:", logits.shape)

## 7. Flattening and Fully Connected Layers

Convolution layers preserve a spatial grid. `flatten` changes each example from `(batch, channels, height, width)` into `(batch, features)`. A linear layer then combines those learned features into one score per genre. The scores are called logits.

In [ ]:
class GenreCNN(nn.Module):
    def __init__(self, class_count):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.35), nn.Linear(128, class_count))

    def forward(self, inputs):
        return self.classifier(self.features(inputs))

model = GenreCNN(len(dataset.classes)).to(device)
print(model)
print("Forward output:", model(images.to(device)).shape)

## 8. Assemble the Full CNN Model

The model has three convolution blocks. Batch normalization stabilizes activations, ReLU adds non-linearity, and max pooling reduces the grid. Adaptive average pooling makes the classifier independent of the exact remaining image size. Dropout randomly removes some training activations to reduce overfitting.

## 9. Define the Loss Function and Optimizer

Cross-entropy compares the predicted class distribution with the true class. Adam changes weights using gradients and adaptive step sizes. The scheduler lowers the learning rate when validation loss stops improving.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5)

def run_epoch(loader, training=False):
    model.train(training)
    total_loss = 0.0
    correct = 0
    total = 0
    for batch_images, batch_labels in loader:
        batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
        if training:
            optimizer.zero_grad()
        logits = model(batch_images)
        loss = criterion(logits, batch_labels)
        if training:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * batch_labels.size(0)
        correct += (logits.argmax(1) == batch_labels).sum().item()
        total += batch_labels.size(0)
    return total_loss / total, correct / total

## 10. Train the CNN with a Mini-Batch Loop

Each training step computes logits, measures loss, backpropagates gradients, and updates parameters. Validation runs without updates. The best validation checkpoint is kept so the final test score uses the most generalizable model.

In [ ]:
history = {"train_loss": [], "validation_loss": [], "train_accuracy": [], "validation_accuracy": []}
best_validation = 0.0
for epoch in range(1, 31):
    train_loss, train_accuracy = run_epoch(train_loader, training=True)
    with torch.no_grad():
        validation_loss, validation_accuracy = run_epoch(validation_loader)
    scheduler.step(validation_loss)
    for key, value in [("train_loss", train_loss), ("validation_loss", validation_loss), ("train_accuracy", train_accuracy), ("validation_accuracy", validation_accuracy)]:
        history[key].append(value)
    if validation_accuracy > best_validation:
        best_validation = validation_accuracy
        torch.save(model.state_dict(), "gtzan_genre_cnn.pt")
    print(f"Epoch {epoch:02d}: train_acc={train_accuracy:.3f}, val_acc={validation_accuracy:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train"); axes[0].plot(history["validation_loss"], label="validation"); axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(history["train_accuracy"], label="train"); axes[1].plot(history["validation_accuracy"], label="validation"); axes[1].set_title("Accuracy"); axes[1].legend()
plt.show()

## 11. Evaluate Accuracy and Confusion Matrix

The test set is used once after model selection. A confusion matrix shows which genres are confused with one another, which is more informative than accuracy alone.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

model.load_state_dict(torch.load("gtzan_genre_cnn.pt", map_location=device))
model.eval()
true_labels, predicted_labels = [], []
with torch.no_grad():
    for batch_images, batch_labels in test_loader:
        predictions = model(batch_images.to(device)).argmax(1).cpu()
        true_labels.extend(batch_labels.numpy())
        predicted_labels.extend(predictions.numpy())
print(classification_report(true_labels, predicted_labels, target_names=dataset.classes, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(true_labels, predicted_labels), display_labels=dataset.classes).plot(xticks_rotation=45, cmap="Blues")
plt.tight_layout(); plt.show()

sample_image, sample_label = test_set[0]
with torch.no_grad():
    probabilities = model(sample_image.unsqueeze(0).to(device)).softmax(1)[0]
print("True:", dataset.classes[sample_label], "Predicted:", dataset.classes[probabilities.argmax().item()])

## 12. Visualize Feature Maps and Learned Filters

Early filters often respond to simple edges and textures. Later feature maps combine those responses into patterns useful for separating genres. The plots below expose activations from the first convolution layer and its learned filters.

## Optional: Connect the Same CNN to FMA

FMA stores MP3 files under `raw/fma_small` and labels in the large `raw/fma_metadata/tracks.csv` file. Its `genre_top` field is hierarchical and does not exactly match GTZAN, so inspect the label distribution before training. To reuse this CNN, load a fixed audio window, compute a mel spectrogram with `librosa`, normalize it, and repeat the single channel to three channels (or change the first convolution to accept one channel).